# Orbit Wars — Agent Test Notebook


In [1]:
import math
from statistics import mean
from typing import List, Tuple, Optional

try:
    from kaggle_environments import make
    print('kaggle_environments loaded ✓')
except ImportError:
    make = None
    print('kaggle_environments not found — install with: pip install kaggle-environments>=1.28.0')

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 24.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game_arena
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environment

In [2]:
# ─── Board constants ──────────────────────────────────────────────────────────
CENTER_X = 50.0
CENTER_Y = 50.0
SUN_RADIUS = 10.0
SUN_SAFE_MARGIN = 1.2
MAX_FLEET_SPEED = 6.0
MIN_GARRISON = 8
ORBIT_THRESHOLD = 50.0

# ─── Tuning knobs ─────────────────────────────────────────────────────────────
SAFETY_MARGIN_SHIPS = 3
THREAT_LOOKAHEAD = 20
COMET_OPPORTUNITY_DIST = 35
REINFORCE_THRESHOLD = 0.7
AGGRESSION = 0.55
SUN_BYPASS_ANGLES = [0.25, -0.25, 0.5, -0.5]

In [3]:
# ─── Physics helpers ──────────────────────────────────────────────────────────

def dist(x1, y1, x2, y2):
    return math.hypot(x1 - x2, y1 - y2)

def fleet_speed(num_ships):
    if num_ships <= 1:
        return 1.0
    n = min(max(int(num_ships), 1), 1000)
    ratio = math.log(n) / math.log(1000)
    return 1.0 + (MAX_FLEET_SPEED - 1.0) * (ratio ** 1.5)

def segment_hits_sun(x1, y1, x2, y2):
    r = SUN_RADIUS + SUN_SAFE_MARGIN
    dx, dy = x2 - x1, y2 - y1
    if dx == 0 and dy == 0:
        return dist(x1, y1, CENTER_X, CENTER_Y) <= r
    t = max(0.0, min(1.0, ((CENTER_X - x1)*dx + (CENTER_Y - y1)*dy) / (dx*dx + dy*dy)))
    return dist(x1 + t*dx, y1 + t*dy, CENTER_X, CENTER_Y) <= r

def safe_angle(src_x, src_y, tgt_x, tgt_y):
    direct = math.atan2(tgt_y - src_y, tgt_x - src_x)
    if not segment_hits_sun(src_x, src_y, tgt_x, tgt_y):
        return direct
    for offset in SUN_BYPASS_ANGLES:
        angle = direct + offset
        far_x = src_x + 200 * math.cos(angle)
        far_y = src_y + 200 * math.sin(angle)
        if not segment_hits_sun(src_x, src_y, far_x, far_y):
            return angle
    return None

def is_orbiting(planet):
    return dist(planet[2], planet[3], CENTER_X, CENTER_Y) + planet[4] < ORBIT_THRESHOLD

def predict_pos(src_x, src_y, target, angular_velocity, launch_ships, iterations=6):
    tx, ty = target[2], target[3]
    if not is_orbiting(target):
        return tx, ty
    dx, dy = tx - CENTER_X, ty - CENTER_Y
    r = math.hypot(dx, dy)
    if r < 1e-6:
        return tx, ty
    theta0 = math.atan2(dy, dx)
    t = 0.0
    for _ in range(iterations):
        theta = theta0 + angular_velocity * t
        fx = CENTER_X + r * math.cos(theta)
        fy = CENTER_Y + r * math.sin(theta)
        d = dist(src_x, src_y, fx, fy)
        t = d / max(fleet_speed(launch_ships), 1e-6)
    theta = theta0 + angular_velocity * t
    return CENTER_X + r * math.cos(theta), CENTER_Y + r * math.sin(theta)

def eta(src_x, src_y, tgt_x, tgt_y, ships):
    return dist(src_x, src_y, tgt_x, tgt_y) / max(fleet_speed(ships), 1e-6)

print('Physics helpers loaded ✓')

Physics helpers loaded ✓


In [4]:
# ─── Strategy helpers ─────────────────────────────────────────────────────────

def incoming_threat(planet, fleets, player_id):
    px, py, pr = planet[2], planet[3], planet[4]
    threat = 0.0
    for f in fleets:
        if f[1] == player_id:
            continue
        fx, fy = f[2], f[3]
        d = dist(fx, fy, px, py)
        if d > 60:
            continue
        expected_angle = math.atan2(py - fy, px - fx)
        angle_diff = abs((f[4] - expected_angle + math.pi) % (2 * math.pi) - math.pi)
        if angle_diff < 0.4:
            weight = 1.0 if d < 20 else 0.5
            threat += f[6] * weight
    return threat

def garrison_needed(planet, fleets, player_id):
    threat = incoming_threat(planet, fleets, player_id)
    base = MIN_GARRISON + max(2, planet[6])
    return int(base + threat * 1.2)

def launch_budget(planet, fleets, player_id, fraction=AGGRESSION):
    reserve = garrison_needed(planet, fleets, player_id)
    available = max(0, planet[5] - reserve)
    return int(available * fraction + 0.5)

def ships_to_capture(target, travel_turns, margin=SAFETY_MARGIN_SHIPS):
    return int(target[5] + target[6] * travel_turns + margin)

def target_score(src_x, src_y, target, comet_ids, player_id):
    if target[0] in comet_ids:
        d = dist(src_x, src_y, target[2], target[3])
        if d > COMET_OPPORTUNITY_DIST:
            return -99.0
        return (target[6] + 0.5) / (d + 1.0) - 0.005 * target[5] + 0.3
    d = dist(src_x, src_y, target[2], target[3])
    prod = target[6]
    ships = target[5]
    neutral_bonus = 0.6 if target[1] == -1 else 0.0
    orbit_bonus = 1.2 if is_orbiting(target) else 1.0
    return orbit_bonus * (prod + neutral_bonus) / (d + 1.0) - 0.01 * ships

def my_fleets_targeting(fleets, player_id, target_id, sx, sy, tx, ty):
    total = 0
    for f in fleets:
        if f[1] != player_id:
            continue
        angle_toward_tgt = math.atan2(ty - f[3], tx - f[2])
        diff = abs((f[4] - angle_toward_tgt + math.pi) % (2 * math.pi) - math.pi)
        if diff < 0.3 and dist(f[2], f[3], tx, ty) < 45:
            total += f[6]
    return total

def needs_reinforcement(planet, fleets, player_id):
    if planet[5] <= 0:
        return 0.0
    threat = incoming_threat(planet, fleets, player_id)
    return threat / max(planet[5], 1)

print('Strategy helpers loaded ✓')

Strategy helpers loaded ✓


In [5]:
import importlib
import main_optimized      # Points to main.py (Agent v4)
import elo_1200 # Points to 1200_elo.py (Your new baseline)

# Force a reload ONCE when you run this cell, NOT on every turn of the game
importlib.reload(main_optimized)
importlib.reload(elo_1200)

def agent_v3(obs, conf=None):
    # Call the actual 'agent' function name defined inside main.py
    return main_optimized.agent(obs, conf)

def baseline_agent(obs, conf=None):
    # Call the actual 'agent' function name defined inside 1200_elo.py
    return elo_1200.agent(obs, conf)

print("Agents bridged successfully with correct entry points ✓")

Agents bridged successfully with correct entry points ✓


In [10]:
# ─── Tournament benchmark (Dynamic Map Setup) ─────────────────────────────────

def run_benchmark(num_games=20, base_seed=19747522367):
    if make is None:
        print('kaggle_environments not available.')
        return

    print(f'Running {num_games}-game tournament on procedurally varied maps...\n')

    wins = losses = draws = 0
    v3_scores, base_scores = [], []

    for i in range(num_games):
        # 1. Dynamically vary the seed for every match to alter planet counts & layouts
        current_seed = base_seed + i
        
        # 2. Instantiate a fresh environment instance for this specific match
        env = make('orbit_wars', configuration={'seed': current_seed}, debug=False)

        # 3. Alternate entry seats to remove any player index/color bias
        if i % 2 == 0:
            agents = [agent_v3, baseline_agent]
            vi, bi = 0, 1
        else:
            agents = [baseline_agent, agent_v3]
            vi, bi = 1, 0

        # Run simulation
        env.run(agents)
        final = env.steps[-1]
        
        # Safely extract scores depending on object wrapper type
        vs = final[vi].get('reward', 0) if isinstance(final[vi], dict) else getattr(final[vi], 'reward', 0) or 0
        bs = final[bi].get('reward', 0) if isinstance(final[bi], dict) else getattr(final[bi], 'reward', 0) or 0
        
        if vs is None: vs = 0
        if bs is None: bs = 0
        
        v3_scores.append(vs)
        base_scores.append(bs)

        if vs > bs:
            wins += 1
            tag = 'V8 WIN ✓'
        elif bs > vs:
            losses += 1
            tag = 'BASELINE WIN ✗'
        else:
            draws += 1
            tag = 'DRAW'

        # Print the seed alongside results so you can trace complex maps
        print(f' Game {i+1:02d} (Seed {current_seed}): {tag:16s} v4={vs:4} base={bs:4}')

    wr = wins / num_games * 100
    print(f'\n{"─"*60}')
    print(f'Tournament Summary:')
    print(f'Win Rate : {wr:.1f}% ({wins}W / {losses}L / {draws}D)')
    print(f'Avg Score: v4={mean(v3_scores):.2f} baseline={mean(base_scores):.2f}')
    print(f'Score Gap: {mean(v3_scores) - mean(base_scores):+.2f} per game')
    print(f'{"─"*60}')

    if wr >= 80:
        print('🔥 Dominant — submit to Kaggle!')
    elif wr >= 60:
        print('📈 Solid improvement — keep tuning.')
    else:
        print('⚠️ Needs work — review tuning knobs.')

# Run a 20-game test starting with seed 1000 up to 1019
run_benchmark(num_games=20, base_seed=67676767)

Running 20-game tournament on procedurally varied maps...

 Game 01 (Seed 67676767): DRAW             v4=   0 base=   0
 Game 02 (Seed 67676768): DRAW             v4=   0 base=   0
 Game 03 (Seed 67676769): DRAW             v4=   0 base=   0
 Game 04 (Seed 67676770): DRAW             v4=   0 base=   0
 Game 05 (Seed 67676771): DRAW             v4=   0 base=   0
 Game 06 (Seed 67676772): DRAW             v4=   0 base=   0
 Game 07 (Seed 67676773): DRAW             v4=   0 base=   0
 Game 08 (Seed 67676774): DRAW             v4=   0 base=   0
 Game 09 (Seed 67676775): DRAW             v4=   0 base=   0
 Game 10 (Seed 67676776): DRAW             v4=   0 base=   0
 Game 11 (Seed 67676777): DRAW             v4=   0 base=   0
 Game 12 (Seed 67676778): DRAW             v4=   0 base=   0
 Game 13 (Seed 67676779): DRAW             v4=   0 base=   0
 Game 14 (Seed 67676780): DRAW             v4=   0 base=   0
 Game 15 (Seed 67676781): DRAW             v4=   0 base=   0
 Game 16 (Seed 67676782): 

In [ ]:
# ─── Save a visual replay of a specific complex tournament seed ──────────────

# 💡 TIP: Change this integer to any seed where your agent struggled above!
TARGET_SEED =  67676786 

if make is not None:
    print(f'Generating visual replay for Seed {TARGET_SEED}...')
    env = make('orbit_wars', configuration={'seed': TARGET_SEED}, debug=True)
    env.run([agent_v3, baseline_agent])
    
    html = env.render(mode='html', width=900, height=700)
    with open(f'Seed {TARGET_SEED}.html', 'w') as f:
        f.write(html)
    print(f'Replay saved → replay_v3.html (Seed {TARGET_SEED})')
    
    # Render inline inside the notebook
    env.render(mode='ipython', width=900, height=700)
else:
    print('kaggle_environments not available — skipping replay.')

Generating visual replay for Seed 67676786...
Replay saved → replay_v3.html (Seed 67676786)
